# 08.04 · Time travel: replay and fork

**Module 08 · LangGraph Production Features** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 90–120 min |
| 📋 Prerequisites | Lessons 08.01 (checkpoints) and 08.03 (interrupts). |
| 🔑 Keys | none (optional: `LANGSMITH_API_KEY` to see the branches as traces) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Find** the checkpoint you want in a thread's history (by `next`, `step`, `source`).
2. **Replay** from a checkpoint: re-run everything after it, and know what is and isn't repeated.
3. **Fork**: change state at a past checkpoint with `update_state` and run a different branch, keeping the original.
4. **Trace** branches: compare them in code and tag them so they're findable in LangSmith.

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Use time travel](https://docs.langchain.com/oss/python/langgraph/use-time-travel) | Replay with `invoke(None, checkpoint_config)`, fork with `update_state`, `as_node` |
| [Persistence](https://docs.langchain.com/oss/python/langgraph/persistence) | `get_state_history`, `update_state` and reducers |
| [LangSmith: add metadata and tags](https://docs.langchain.com/langsmith/add-metadata-tags) | Making runs findable and filterable |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2" "langsmith>=0.14.7,<1"

**You should now see** nothing, or *"Note: you may need to restart the kernel to use updated packages."* Restart only if a package was **upgraded** after you had already imported it (Lesson 00.07). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.03).

In [ ]:
from importlib import metadata

for package in ("langgraph", "langsmith",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/08-langgraph-production-features/).
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import the course package (installed editable in Lesson 00.02; src/ fallback otherwise).
try:
    import mastery
except ModuleNotFoundError:
    sys.path.insert(0, str(REPO_ROOT / "src"))
    import mastery
    print("ℹ️ Loaded mastery from src/. Fix once: python -m pip install -e \".[dev]\"")
print("Repo root:", REPO_ROOT)

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY', 'LANGSMITH_API_KEY')
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Step 4: Silence known, harmless noise so real problems stand out.
import os
import warnings

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (Project 08 can reuse Module 06): no usage telemetry.
# Same as Lesson 02.04: langchain-core 1.6.9/1.6.10 Pydantic serializer warning about the injected runtime.
warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

> Everything runs 🧪 offline with Project 08's research graph and scripted models. With `LANGSMITH_API_KEY` and `LANGSMITH_TRACING=true` in `.env`, every run below is also traced (free tier), and Part 4 shows how to find the branches there.

---
## Part 1 · A thread to travel in

In [ ]:
# 🧪 Offline: one full run of the research graph (approve on the first pause).
from langgraph.types import Command
from mastery.graphs.hitl import adrive, arun_until_pause, scripted_reviewer
from mastery.graphs.persistence import acheckpoint_rows
from mastery.graphs.research import build_research_graph, corpus_docs_search, offline_models, offline_web_search
from mastery.graphs.state import memory_checkpointer

agent = build_research_graph(offline_models(), web_search=offline_web_search(), docs_search=corpus_docs_search(),
                             checkpointer=memory_checkpointer())
THREAD = {"configurable": {"thread_id": "travel-1"}}
QUESTION = "How do I keep the battery healthy in winter?"
main = await adrive(agent, {"question": QUESTION}, {**THREAD, "metadata": {"branch": "main"}, "tags": ["m08", "main"]},
                    scripted_reviewer(["approve"]))
print(main["values"]["final"][:260])
print()
for row in await acheckpoint_rows(agent, THREAD):
    print(f"step {row['step']:>2} {row['source']:<6} next={str(row['next']):<22} id={row['checkpoint_id'][-8:]} parent={str(row['parent_id'])[-8:]}")

**You should now see** the start of a battery answer (two bullets with `[source: ...]`) and 8 checkpoints, newest first: step `6` (`next=()`) down to step `-1` (`input`). Each `parent` is the id of the row below it: a straight chain.

`acheckpoint_rows` (course helper) is `aget_state_history` flattened into dicts; `checkpoint_rows` is the sync version.

In [ ]:
# 🧪 Offline: pick checkpoints by what runs NEXT (more robust than counting steps).
history = [s async for s in agent.aget_state_history(THREAD)]
after_plan = next(s for s in history if s.next == ("research",))       # plan done, research not yet
before_approval = next(s for s in history if s.next == ("approval",))
print("after_plan      :", after_plan.values["plan"]["queries"])
print("before_approval :", before_approval.values["draft"][:70].replace("\n", " "), "...")

**You should now see** the two planned queries (the question, and the question + `troubleshooting`) and the start of the draft at the approval checkpoint.

---
## Part 2 · Replay: run again from a past checkpoint

`invoke(None, checkpoint.config)`: `config` contains a `checkpoint_id`, so the run starts **there** instead of at the newest checkpoint. Nodes after it **run again** (new LLM calls, new searches); the old checkpoints stay.

In [ ]:
# 🧪 Offline: replay from after_plan. Research and draft re-run; approval pauses again.
replay = await arun_until_pause(agent, None, after_plan.config)
print("paused again:", bool(replay["interrupts"]))
replay = await arun_until_pause(agent, Command(resume="approve"), THREAD, printer=None)   # resume the NEWEST
print("same answer as the main run:", replay["values"]["final"] == main["values"]["final"])
print("checkpoints now:", len([s async for s in agent.aget_state_history(THREAD)]))

**You should now see** `✅ research ...`, `✅ draft ...`, `⏸️  paused → ...`, then `paused again: True`, `same answer as the main run: True` and `checkpoints now: 13` (8 + 5 new).

### What just happened?

The offline models are deterministic, so the replay produced the same answer. With a real model, a replay is a **new sample**: different words, maybe a different result. Two details:

- The resume used `THREAD` (no `checkpoint_id`) so it continues from the **newest** checkpoint, the replayed pause. Resuming with `after_plan.config` would start from the old checkpoint again.
- Interrupts **after** the replay point fire again (the human has to approve again). That's what you want: a replayed draft is a new draft.

---
## Part 3 · Fork: change the past, keep both futures

`update_state(checkpoint.config, values, as_node="plan")` writes a **new** checkpoint whose parent is that old checkpoint, as if the `plan` node had returned `values`. The graph continues from it as usual (`next` = what follows `plan`). The original branch is untouched.

In [ ]:
# 🧪 Offline: fork after_plan with a different plan (mounting instead of battery).
new_plan = {"queries": ["mount the station outdoors safely"], "outline": ["Where to mount", "How to mount"]}
fork_cfg = await agent.aupdate_state(after_plan.config, {"plan": new_plan}, as_node="plan")
print("fork checkpoint:", fork_cfg["configurable"]["checkpoint_id"][-8:])

FORK_META = {"metadata": {"branch": "fork-1", "forked_from": after_plan.config["configurable"]["checkpoint_id"]},
             "tags": ["m08", "fork"]}
await arun_until_pause(agent, None, {**fork_cfg, **FORK_META}, printer=None)
fork = await arun_until_pause(agent, Command(resume="approve"), {**THREAD, **FORK_META}, printer=None)
print(fork["values"]["final"])

**You should now see** a fork checkpoint id and an answer about **mounting** (`guides/mounting.md` sources) instead of battery care: the same question, a different plan, a different branch.

> `as_node` matters: it decides which edges run next. `as_node="plan"` → `research` runs next. Without it, LangGraph uses the last node that ran before that checkpoint. Values go through **reducers**: a list with `operator.add` (like `findings`) gets your items **appended**, not replaced.

In [ ]:
# 🧪 Offline: the thread is now a tree. Print it from the input checkpoint down.
snaps = [s async for s in agent.aget_state_history(THREAD)]
children = {}
for s in snaps:
    parent = s.parent_config["configurable"]["checkpoint_id"] if s.parent_config else None
    children.setdefault(parent, []).append(s)


def show(parent=None, depth=0):
    for s in sorted(children.get(parent, []), key=lambda s: s.config["configurable"]["checkpoint_id"]):
        branch = s.metadata.get("branch", "")
        print("   " * depth + f"└ step {s.metadata['step']} {s.metadata['source']:<6} next={s.next} {branch}")
        show(s.config["configurable"]["checkpoint_id"], depth + 1)


show()

**You should now see** a chain from `step -1 input` down to `step 2 loop next=('research',)`, where the tree splits into **three** children:

- the main branch (`main`, ending at `step 6 loop next=()`);
- the replay: `step 3 fork` (measured: replaying from an old checkpoint writes a checkpoint with `source='fork'`), no label because that call had no metadata;
- the fork: `step 3 update` (written by `update_state`), then `fork-1` checkpoints down to `step 7 loop next=()`.

Every checkpoint written by a run with `metadata={...}` carries it, so `branch` and `forked_from` are stored **in the thread**: you can find a branch later without remembering ids.

In [ ]:
# 🧪 Offline: compare the two final answers side by side.
def sources(state):
    return sorted({f["source"] for f in state["findings"]})

print("main  :", sources(main["values"]), "|", main["values"]["plan"]["queries"][0])
print("fork-1:", sources(fork["values"]), "|", fork["values"]["plan"]["queries"][0])

**You should now see** different source lists (the battery FAQ for `main`, mounting/returns documents for `fork-1`) and the different first queries.

---
## Part 4 · Find the branches in LangSmith

With tracing on, each `ainvoke`/`astream` call is a trace. The `metadata` and `tags` you passed are on the trace, so in LangSmith you can filter `metadata.branch = fork-1` or the tag `fork`. Studio's thread view shows the same fork as a branch you can click (the fork button does what Part 3 did).

In [ ]:
# 🌐 Network (LangSmith API, no model tokens): list this notebook's recent fork traces.
import os

if os.environ.get("LANGSMITH_API_KEY") and os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    from langsmith import Client

    project = os.environ.get("LANGSMITH_PROJECT", "default")
    runs = list(Client().list_runs(project_name=project, filter='has(tags, "fork")', is_root=True, limit=5))
    print(f"{len(runs)} fork traces in project {project!r}")
    for run in runs:
        print(" ", run.name, run.start_time, (run.extra or {}).get("metadata", {}).get("branch"))
else:
    print("⏭️ Skipped: set LANGSMITH_API_KEY and LANGSMITH_TRACING=true to trace (see Lesson 00.05).")

**You should now see** (with tracing) up to 5 `research_agent` traces with `fork-1`; traces can take a few seconds to appear, so re-run the cell if the count is 0. Without LangSmith: `⏭️ Skipped`.

---
## ⚠️ Pitfalls and troubleshooting (Windows Git Bash included)

| Symptom | Cause | Fix |
|---|---|---|
| Resuming jumps back to an old point | Resumed with a config that still has a `checkpoint_id` | Resume with the plain thread config |
| The fork runs the wrong node next | `update_state` without `as_node` | `as_node="<node whose output you're replacing>"` |
| A list in state doubled after `update_state` | The key has a reducer (`operator.add`) | Send only new items, or use `Overwrite` (Module 07) |
| A replay gives a different answer | Real models sample; nodes re-run | Expected. Fix temperature/seed, or fork instead of replay to compare |
| Replay sent the email again | Side effects after the replay point re-run | Idempotency keys (Lesson 08.02) |
| `aget_state_history` is empty | Different `thread_id`, or `InMemorySaver` after a restart | Same id; SQLite for history across restarts |

## ⚠️ Production notes

> ⚠️ **Production note: time travel is a debugging and review tool.**
> It's great for "what if the plan had been X?", reproducing a bug from a user's thread, and A/B-ing prompts on the exact same state. Don't build product logic that depends on rewinding user threads; branches multiply storage.

> ⚠️ **Production note: record why a branch exists.**
> Put `branch`, `forked_from` and the author in `metadata` on every fork. Six weeks later, nobody remembers which checkpoint id was the "good" one.

## 🏋️ Exercises

1. **(Easy)** Fork `before_approval` with a hand-written draft (`as_node="draft"`) and approve it. What's in `final`?
2. **(Medium)** Fork `after_plan` with an **empty** queries list. What happens, and why? (Hint: look at `Plan` in `mastery/graphs/research.py`.)
3. **(Medium)** Write `branch_finals(agent, thread)` that returns `{branch_label: final_text}` for every finished leaf of the tree.
4. **(Stretch)** Repeat Part 3 with live models (`get_model("fast")` for all three roles). Fork twice with two different plans and compare the answers.

<details>
<summary><b>Show solutions</b></summary>

**1.**
```python
cfg2 = await agent.aupdate_state(before_approval.config, {"draft": "My own short answer."}, as_node="draft")
await arun_until_pause(agent, None, cfg2, printer=None)
out = await arun_until_pause(agent, Command(resume="approve"), THREAD, printer=None)
print(out["values"]["final"])     # your text + the Sources line
```

**2.** The state accepts it (it's a dict), and the research subgraph finds nothing for zero queries, so the draft says it found nothing relevant. The `Plan` model (1–4 queries) is only enforced when the **planner** produces the plan. Validate in the node that reads it, or validate before `update_state`: `Plan.model_validate(new_plan)` raises.

**3.**
```python
async def branch_finals(agent, thread):
    snaps = [s async for s in agent.aget_state_history(thread)]
    return {s.metadata.get("branch", s.config["configurable"]["checkpoint_id"][-6:]): s.values["final"]
            for s in snaps if not s.next and "final" in s.values}
```

**4.** Use the 💸 pattern from Lesson 08.03 Part 6 to build `live`, then the same `aupdate_state` + `arun_until_pause` calls with `FORK_META` labels `fork-a` and `fork-b`.
</details>

## ✅ Checkpoint

- [ ] I can find a checkpoint by `next` and print a thread's history.
- [ ] I replayed from a checkpoint and know which nodes re-run.
- [ ] I forked with `update_state(..., as_node=...)` and got a different branch.
- [ ] I labelled branches with metadata/tags and can print the thread as a tree.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Read [Use time travel](https://docs.langchain.com/oss/python/langgraph/use-time-travel) and note in [`docs/research-log.md`](../../docs/research-log.md): what the docs say about checkpoint `source` values (this lesson measured `fork` for a replay and `update` for `update_state`), and how Studio exposes forking.

## Summary

- History = all checkpoints; pick by `next`. `invoke(None, checkpoint.config)` replays from there.
- Replays re-run nodes (and interrupts) after the point; resume with the plain thread config.
- `update_state(cfg, values, as_node=...)` forks: a new child checkpoint, original branch kept, reducers applied.
- `metadata`/`tags` on each run make branches findable in the thread and in LangSmith.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 08.05 · Subgraphs** → [`08-05-subgraphs.ipynb`](08-05-subgraphs.ipynb).

[Module 08 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)